# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [4]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.496500,0.527291,0.637203,0.806670,...,0.3,0.815075,0.788450,0.867595,0.911292,0.783514,0.868510,0.784140,0.618609,0.543036
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.340099,0.303278,0.512236,0.734278,...,0.7,0.686883,0.686778,0.594025,0.698730,0.695484,0.772514,0.666212,0.572044,0.446302
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.127990,0.192701,0.330117,0.533609,...,0.5,0.580053,0.607686,0.523752,0.600009,0.614942,0.717012,0.570198,0.631260,0.440385
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,-1.249994,-1.823374,-0.834575,0.001354,...,0.0,0.002635,0.002627,0.013206,0.025528,0.000000,0.005494,0.063219,0.000000,0.076485
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,-0.090382,-0.011299,0.200067,0.386963,...,0.3,0.397511,0.457883,0.364956,0.265409,0.452112,0.482416,0.418672,0.245866,0.244423


In [5]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,pca_knn_proba,ridge_logit,truncatedsvd_linear_svc_logit,truncatedsvd_ridge_logit,sgdclassifier_proba,...,truncatedsvd_rbfsampler_knn_proba,truncatedsvd_logistic_regression_proba,truncatedsvd_nystroem_logistic_regression_proba,truncatedsvd_nystroem_sgdclassifier_proba,truncatedsvd_rbfsampler_logistic_regression_proba,truncatedsvd_rbfsampler_sgdclassifier_proba,voting_lgbm_and_cat_and_xgb_and_hist_proba,voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier,voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba,stacking_voting_lgbm_and_cat_and_xgb_and_hist_proba_and_voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier_and_voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba_and_truncatedsvd_linear_svc_logit
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,-1.272709,-1.721325,-0.872993,0.005473,...,0.0,0.006534,0.003712,0.009642,0.010313,0.000000,0.019849,0.061327,0.000000,0.076454
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,-0.626960,-0.919863,-0.289991,0.054146,...,0.1,0.063555,0.110500,0.044563,0.145672,0.047794,0.012173,0.144461,0.102869,0.089535
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,-1.335019,-1.809246,-0.917667,0.003687,...,0.0,0.004404,0.003123,0.008417,0.006856,0.000000,0.010304,0.057929,0.000000,0.076454
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.190133,0.328363,0.387739,0.676445,...,0.3,0.670446,0.712745,0.648495,0.504756,0.637237,0.444558,0.628242,0.183451,0.215908
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.535153,0.644014,0.670147,0.852891,...,0.8,0.835307,0.855796,0.796202,0.823601,0.787922,0.939648,0.787450,0.823541,0.643512


# Machine Learning

In [6]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": 5000
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(**params)
        )
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-25 14:56:09,788] A new study created in memory with name: no-name-5e01c161-09b8-4f51-a366-0523f0364c41
Best trial: 8. Best value: 0.948775:   2%|▉                                            | 1/50 [00:14<12:03, 14.77s/it]

[I 2026-05-25 14:56:24,553] Trial 8 finished with value: 0.9487749635788969 and parameters: {'solver': 'saga', 'C': 0.00010633347666531953, 'l1_ratio': 0.027101924790274734, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.005964200676938382}. Best is trial 8 with value: 0.9487749635788969.


Best trial: 8. Best value: 0.948775:   4%|█▊                                           | 2/50 [00:18<06:34,  8.23s/it]

[I 2026-05-25 14:56:28,204] Trial 2 finished with value: 0.9296732674773158 and parameters: {'solver': 'saga', 'C': 15.462008893041569, 'l1_ratio': 0.07059254315069141, 'class_weight': None, 'fit_intercept': False, 'tol': 0.006255218777310399}. Best is trial 8 with value: 0.9487749635788969.


Best trial: 7. Best value: 0.949169:   6%|██▋                                          | 3/50 [00:19<03:58,  5.07s/it]

...

[I 2026-05-25 15:52:10,041] Trial 21 pruned. 


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Best trial: 40. Best value: 0.949703:  98%|█████████████████████████████████████████▏| 49/50 [57:17<04:14, 254.34s/it]

[I 2026-05-25 15:53:26,869] Trial 28 pruned. 


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Best trial: 40. Best value: 0.949703: 100%|█████████████████████████████████████████| 50/50 [1:19:54<00:00, 95.88s/it]

[I 2026-05-25 16:16:03,869] Trial 13 finished with value: 0.9488721724343595 and parameters: {'solver': 'saga', 'C': 85.00201927959763, 'l1_ratio': 0.06823211381107741, 'class_weight': None, 'fit_intercept': True, 'tol': 1.5122132903929594e-06}. Best is trial 40 with value: 0.9497027473076102.
Best trial score:
0.9497027473076102

Best params:
{'solver': 'saga', 'C': 5.330159475877343e-05, 'l1_ratio': 0.7719772792871329, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.0004370254875869294}


In [7]:
stacking_lg = make_pipeline(
    StandardScaler(),
    LogisticRegression(**study.best_params, max_iter=5000)
).fit(X_train, y_train.PitNextLap)

# Submission

In [8]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [9]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test)[:, 1]

In [10]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [11]:
X_train.columns.tolist()

['lgbm_proba',
 'cat_proba',
 'xgb_proba',
 'hist_proba',
 'lg_proba',
 'pca_knn_proba',
 'ridge_logit',
 'truncatedsvd_linear_svc_logit',
 'truncatedsvd_ridge_logit',
 'sgdclassifier_proba',
 'truncatedsvd_lgbm_proba',
 'truncatedsvd_knn_proba',
 'truncatedsvd_nystroem_knn_proba',
 'truncatedsvd_rbfsampler_knn_proba',
 'truncatedsvd_logistic_regression_proba',
 'truncatedsvd_nystroem_logistic_regression_proba',
 'truncatedsvd_nystroem_sgdclassifier_proba',
 'truncatedsvd_rbfsampler_logistic_regression_proba',
 'truncatedsvd_rbfsampler_sgdclassifier_proba',
 'voting_lgbm_and_cat_and_xgb_and_hist_proba',
 'voting_and_lg_and_ridge_and_truncatedsvd_ridge_and_sgdclassifier_and_truncatedsvd_logistic_regression_and_truncatedsvd_nystroem_logistic_regression_and_truncatedsvd_nystroem_sgdclassifier_and_truncatedsvd_rbfsampler_logistic_regression_and_truncatedsvd_rbfsampler_sgdclassifier',
 'voting_truncatedsvd_nytroem_knn_and_truncated_rbfsampler_knn_and_truncatedsvd_knn_and_pca_knn_proba',
 's